# Notebook 01 — Data Pipeline

**Goal:** Load, filter, standardize, and clip all tracking + reference data into clean outputs for downstream notebooks.

**Outputs:**
- `outputs/tracking_clipped.parquet` — EDGE rusher tracking rows, frame_offset 0–25
- `outputs/edge_pff.csv` — PFF scouting rows for EDGE rushers on passing plays

**Run order:** Must run before any other notebook.

## Section 1 — Imports & Paths

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

REPO_ROOT = Path.cwd().parent        # notebooks/ -> repo root
DATA_DIR  = REPO_ROOT                # CSVs live here (downloaded from Kaggle)
OUT_DIR   = REPO_ROOT / "outputs"

print("Repo root:", REPO_ROOT)
print("Week CSVs found:", len(list(DATA_DIR.glob("week*.csv"))))

Repo root: c:\Users\parik\Documents\Baltimore Ravens Data Scientist Project\nfl-big-data-bowl-2023
Week CSVs found: 8


## Section 2 — Load Tracking Data (Weeks 1–8)

In [ ]:
weeks = []
# Weeks start at 1 and stop at 8 (inclusive)
for i in range(1, 9):
    df = pd.read_csv(DATA_DIR / f"week{i}.csv")
    weeks.append(df)

tracking = pd.concat(weeks, ignore_index=True)

print(tracking.shape)
print(tracking.columns.tolist())
print(tracking['event'].value_counts(dropna=False).head(10))

(8314178, 16)
['gameId', 'playId', 'nflId', 'frameId', 'time', 'jerseyNumber', 'team', 'playDirection', 'x', 'y', 's', 'a', 'dis', 'o', 'dir', 'event']
None                         7673352
ball_snap                     196236
pass_forward                  173604
autoevent_ballsnap             86641
autoevent_passforward          85882
play_action                    45471
run                            10902
qb_sack                        10373
pass_arrived                    8441
autoevent_passinterrupted       4623
Name: event, dtype: int64


## Section 3 — Load Reference Tables

In [ ]:
plays = pd.read_csv(DATA_DIR / f"plays.csv")   # plays.csv
players = pd.read_csv(DATA_DIR / f"players.csv")   # players.csv
pff = pd.read_csv(DATA_DIR / f"pffScoutingData.csv")   # pffScoutingData.csv

print(plays.shape)
print(players.shape)
print(pff.shape)
print(pff.columns.tolist())

(8557, 32)
(1679, 7)
(188254, 15)
['gameId', 'playId', 'nflId', 'pff_role', 'pff_positionLinedUp', 'pff_hit', 'pff_hurry', 'pff_sack', 'pff_beatenByDefender', 'pff_hitAllowed', 'pff_hurryAllowed', 'pff_sackAllowed', 'pff_nflIdBlockedPlayer', 'pff_blockType', 'pff_backFieldBlock']


## Section 4 — Filter to Passing Plays

In [ ]:
# Drop any rows that are non passing plays (since I am only evaluating their pass rushing ability)
passing_plays = plays[plays['passResult'].isin(['C', 'I', 'S', 'IN', 'R'])]

print(passing_plays.shape)
print(passing_plays['passResult'].value_counts())

(8557, 32)
C     4620
I     2755
S      543
R      449
IN     190
Name: passResult, dtype: int64


## Section 5 — Filter PFF to EDGE Pass Rushers

In [ ]:
# Keeps players that were supposed to rush the passer on a given play that were aligned at the EDGE position
edge_pff = pff[
    (pff['pff_role'] == 'Pass Rush') &
    (pff['pff_positionLinedUp'].isin(['LEO', 'REO', 'LOLB', 'ROLB', 'LE', 'RE']))   # 6 codes to denote EDGE rushers according to PFF
]

# keep only edge rush attempts on passing plays ONLY
edge_pff = edge_pff.merge(passing_plays[['gameId', 'playId']],
                           on=['gameId', 'playId'],
                           how='inner')

print(edge_pff.shape)
print(edge_pff['pff_positionLinedUp'].value_counts())
print(f"Unique players: {edge_pff['nflId'].nunique()}")

(22254, 15)
LEO     4601
REO     4226
ROLB    4040
LOLB    3838
LE      3182
RE      2367
Name: pff_positionLinedUp, dtype: int64
Unique players: 564


## Section 6 — Coordinate Standardization

In [12]:
# Make all plays attack the same direction (increasing x)
# Plays where the offense attacks left are mirrored (120-x)
# 120 comes from 10 * 2 yards from both endzones + 100 from the length of the field

tracking['x_std'] = np.where(tracking['playDirection'] == 'left', 120-tracking['x'], tracking['x'])

# Verify: mean x_std should look similar for both directions (no longer mirrored)
print(tracking.groupby('playDirection')['x_std'].mean())

playDirection
left     60.542008
right    58.839910
Name: x_std, dtype: float64


## Section 7 — Build Snap Frame Lookup & Clip Tracking

In [14]:
# Filter tracking data to pass plays only
tracking_passes = tracking.merge(passing_plays[['gameId', 'playId']],
                                  on=['gameId', 'playId'],
                                  how='inner')

# get the frameId where the ball is snapped for each play
snap_frames = (tracking_passes[tracking_passes['event'] == 'ball_snap']
               .groupby(['gameId', 'playId'])['frameId']
               .first()
               .reset_index()
               .rename(columns={'frameId': 'snap_frame'}))

# Attach the snap_frame as a column to every tracking row for the same play
tracking_passes = tracking_passes.merge(snap_frames,
                                  on=['gameId', 'playId'],
                                  how='left')

# Computes how many frames after the snap each row is
tracking_passes['frame_offset'] = tracking_passes['frameId'] - tracking_passes['snap_frame']

# We only keep track of 0 to 2.5 seconds from the time the ball is snapped (0 to 25 frames, since we are at 10 fps)
tracking_clipped = tracking_passes[(tracking_passes['frame_offset'] >= 0) & (tracking_passes['frame_offset'] <= 25)]

print(tracking_clipped.shape)
print(tracking_clipped['frame_offset'].min())
print(tracking_clipped['frame_offset'].max())

(5047120, 19)
0.0
25.0


## Section 8 — Save Outputs & Final Verification

In [15]:
# Ensure output directory exists
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save clipped tracking data (parquet for fast loading, CSV for inspection)
tracking_clipped.to_parquet(OUT_DIR / "tracking_clipped.parquet", index=False)
tracking_clipped.to_csv(OUT_DIR / "tracking_clipped.csv", index=False)

# Save EDGE PFF scouting data
edge_pff.to_csv(OUT_DIR / "edge_pff.csv", index=False)

# Final verification
print(tracking_clipped.shape)
print(edge_pff.shape)
print(f"Unique EDGE players: {edge_pff['nflId'].nunique()}")

(5047120, 19)
(22254, 15)
Unique EDGE players: 564
